# Mathematische OSS in der Praxis
**Logik, Optimierung, Graphentheorie und mehr mit SageMath**

**Manfred Scheucher**  
Grazer Linuxtage 2026

<img src="figs/arrangement.png" width="30%">

Slides: <a href="https://github.com/manfredscheucher/talk-glt26">https://github.com/manfredscheucher/talk-glt26</a>

---
# Was erwartet mich in diesem Vortrag?

* Slides in Jupyter Notebook — Markdown und Code gemischt
  * sonst LaTeX Beamer oder [IPE Drawing Editor](https://ipe.otfried.org/) (Vektorgrafiken oder ganze Slides)
* viele Beispiele
* schöne Bilder
* Code-Snippets in Python-Syntax
* Python-Kenntnisse hilfreich, aber nicht notwendig

---
# SageMath 

* Freie Software zur Lösung mathematischer Probleme
* vereint Stärken vieler Open-Source-Libraries
  * **PARI/GP, FLINT, NTL** — Zahlentheorie (elliptische Kurven, Polynome, Gitter)
  * **GAP, Singular** — Gruppen- und Ringtheorie, Gröbner-Basen
  * **Maxima, SymPy** — symbolische Algebra, Analysis
  * **GLPK** — lineare/ganzzahlige Optimierung (LP/IP/MILP)
  * **NumPy/SciPy** — numerisch
  * **NetworkX, Boost, Nauty** — Graphen, Automorphismen
  * **Matplotlib, Three.js, Jmol** — 2D/3D-Plots
  * ...
* einheitliches Python-Interface
  * große Teile in Cython (Performance, C/C++-Anbindung)
  * auch R, Octave, Mathematica, ... nutzbar
* 150+ Seiten Dokumentation mit Beispielen
    * [https://doc.sagemath.org/](https://doc.sagemath.org/)
* Installation: conda-forge, pip (Modularisierung in Arbeit)
    * [https://doc.sagemath.org/html/en/installation/](https://doc.sagemath.org/html/en/installation/)

---
## Warum nicht einfach C/C++ oder Python?

**C/C++ — Integer Overflow:**
```c
int x = 1;
for (int i = 0; i < 100; i++) x *= 2;
// → 0  (Overflow!)
```

**C++ mit GMP** (GNU Multiple Precision Arithmetic Library) — beliebig große Ganzzahlen:
```c
#include <gmpxx.h>
mpz_class x = 1;
for (int i = 0; i < 100; i++) x *= 2;
// → 1267650600228229401496703205376
```

**Python** — big integers out of the box:
```python
2**100  # → 1267650600228229401496703205376
```

**Aber — Gleitkomma:**
```c
#include <cmath>
float  x = sqrt(2.0f); printf("%.20f\n", x*x); // → 1.99999988079071044922
double x = sqrt(2.0);  printf("%.20f\n", x*x); // → 2.00000000000000044409
```

```python
import math
x = math.sqrt(2.0)
print(x*x)  # → 2.0000000000000004
```


Irrationale Zahlen sind zwar nicht in Floats darstellbar — aber hier wirds spannend!

## Irrationale Zahlen: Algebraisch vs. Transzendent

**Algebraische Zahlen** — Nullstellen von Polynomen mit ganzzahligen Koeffizienten:
- $\sqrt{2}$: Nullstelle von $x^2 - 2$
- $\sqrt[3]{5}$: Nullstelle von $x^3 - 5$
- $\sqrt{2}+1$: Nullstelle von $(x-1)^2 - 2=(x^2-2x+1)-2=x^2-2x-1$

Durch Repräsentation als Polynom ist **exaktes Rechnen** möglich!

**Beispiel:**
```python
sqrt(6) / sqrt(2)   # → sqrt(3)
```

Intern: $z = x/y$ mit $x^2 = 6$, $y^2 = 2$ $\;\Rightarrow\;$ $z^2 = x^2/y^2 = 6/2 = 3$, also Minimalpolynom $z^2 - 3$.

**Komplizierteres Beispiel:** $x = \sqrt{2}+\sqrt{3}$

$$\begin{align}
x - \sqrt{2} &= \sqrt{3} \\
(x-\sqrt{2})^2 &= 3 \\
x^2 - 2x\sqrt{2} + 2 &= 3 \\
x^2 - 1 &= 2x\sqrt{2} \\
(x^2-1)^2 &= 8x^2 \\
x^4 - 10x^2 + 1 &= 0
\end{align}$$

```python
(sqrt(2)+sqrt(3)).minpoly()  # → x^4 - 10*x^2 + 1
```


**Transzendente Zahlen** — kein solches Polynom existiert für $\pi$, $e$, $\sin(1)$, ...

- aber: bekannte Identitäten sind in Sage symbolisch vorhanden:
```python
sin(pi)    # → 0
sin(pi/2)  # → 1
```
- Sage erkennt sogar Grenzwerte/Folgen/Reihen und formt um!
```python
n = var('n')
limit((1 + 1/n)^n, n=oo)  # → e
```

```python
n, x = var('n x')
sum(x^n / factorial(n), n, 0, oo)  # → e^x
```

*Anmerkung*: Kommerzielle Tools wie Wolfram Mathematica/Alpha können das natürlich auch — aber Sage ist OSS: wer will, kann bis ganz unten lesen und alles verstehen.

---
# Basics

Wie verwendet man "sage"?
* Jupyter Notebook `sage --notebook` oder `sage -n` # <-- im gesamten talk!
* CommandLineInterface `sage`
* Script mit Parameter `sage script.sage`

In [ ]:
print("Hallo Welt!") # Python Input/Output
print("Hallo Graz!")

In [ ]:
"Hallo Welt!" 
"Hallo Graz!" # Jupyter Notebook zeigt letztes an

In [ ]:
def f(x): return x*x
print([f(x) for x in range(10)]) # Funktionen, List Comprehension, etc.

In [ ]:
def my_generator():
    s = 'hallo'
    while True:
        yield s
        s += 'X'

g = my_generator()
for i in range(10): print(i,next(g)) # Generatoren

**Übersicht über Objekte und Methoden**

In [ ]:
G=Graph()
G?

In [ ]:
#G.<tabulator>     ->  Liste von Attributen und Methoden
G.is_connected?

**Zeit messen in Jupyter**

In [ ]:
%%time 
def fib(x): return x if x<=1 else fib(x-1)+fib(x-2)
print([fib(x) for x in range(32)]) 

In [ ]:
%%time
@cached_function 
def fib(x): return x if x<=1 else fib(x-1)+fib(x-2)
print([fib(x) for x in range(32)]) # Dynamische Programmierung: Zwischenergebnisse speichern

**Command-Line Parameter**
* Ausführung auch via commandline: `sage script.sage` bzw `time sage script.sage`
* Parameter `argv` bzw `argparse`

```python
# https://docs.python.org/3/library/argparse.html
parser.add_argument('filename') # mandatory filename
parser.add_argument('-c', '--count', type=int) # optional integer
parser.add_argument('-v', '--verbose', action='store_true') # optional flag
```

**Multithreading**

In [ ]:
%%time 
import time
def f(i): 
    time.sleep(float(0.1)) 
    return i
print([f(i) for i in range(50)]) # 5 Sekunden auf 1 CPU (single-threaded)

In [ ]:
%%time 
@parallel
def f(i): 
    time.sleep(float(0.1)) 
    return i
print(list(f(range(50)))) # 0.1 Sekunden (parallelisiert in Jupyter)

Sideremarks: 
* in diesem Beispiel braucht selbst `print(list(f(range(1000))))` nur 0.1 sec da kein busy wait!
* für Sage-Script `multiprocessing.Pool.map` verwenden, siehe [pool_example.sage](pool_example.sage)

In [ ]:
# Zeitmessen und Parallelisierung in CL
!sage pool_example.sage 

---
## Zahlentheorie & Kryptographie

In [ ]:
x = 1337
print(x.is_prime())
print(x.factor())
print(next_prime(x))
# Liste von Methoden durch Eingabe von: x.<tabulator>

**Modulare Arithmetik** (RSA u.a. Cryptosysteme):

$$2^{12} \mod 17 = \;?$$


In [ ]:
power_mod(2, 12, 17)
# 2^12 = 2^(4*3) = (2^4)^3 = 16^3 = (-1)^3 = -1 = 16 mod 17

Ausführliche Funktionen und Doku zu **ECC** (Elliptic Curve Cryptography):

<a href="https://doc.sagemath.org/html/en/reference/arithmetic_curves/index.html">https://doc.sagemath.org/html/en/reference/arithmetic_curves/index.html</a>

---
## Kurvendiskussion


Klassische Matura-Aufgaben

In [ ]:
# Beispiel 1
x = var('x')
f = 1-x^2
print(f) # Textuelle Ausgabe
f.show() # Formatierte Ausgabe

In [ ]:
print("f'(x) =", f.diff()) # Symbolische Ableitung
print("F(x)  =", f.integrate(x)) # Stammfunktion

In [ ]:
f.roots() # nullstellen mit multiplizitäten

In [ ]:
f.find_local_maximum(-1,1) # numerische suche nach lokalem maximum,  zuerst y wert, dann x wert

In [ ]:
f.diff().roots() # extremwerte symbolisch

In [ ]:
# Plot: Fläche unter Kurve
p  = plot(f, (x, -2, 2))
p += plot(f, (x, -1, 1), fill=True, fillcolor="blue", fillalpha=0.25)
p.show()

In [ ]:
# Flächenbestimmung
I = integrate(f, x, -1, 1)
print(f"Fläche: {I} ≈ {float(I)}") # Rechnung symbolisch, Ergebnis numerisch umwandeln

In [ ]:
# Beispiel 2: Elementare Funktionen (Sinus, Kosinus,...)
f = x*sin(x)

p  = plot(f, (x, -2*pi, 2*pi))
p += plot(f, (x, pi/4, pi/2), fill=True, fillcolor="blue", fillalpha=0.25)
p.show()

In [ ]:
f.find_local_maximum(0,pi) # zuerst y wert, dann x wert

In [ ]:
# Flächenbestimmung
I = integrate(f, x, pi/4, pi/2)
print(f"Fläche: {I} ≈ {float(I)}") # Rechnung symbolisch, Ergebnis numerisch umwandeln

In [ ]:
# beliebige Präzision (Anzahl Bits)
for prec in [30,100,300,1000]:
    R = RealField(prec)
    print(f"{prec} \t-> {R(I)}")

In [ ]:
# Plot: Fläche zwischen Kurven 
f = x*sin(x)
g = -x*sin(x*x)

x1 = (f-g).find_root(2,2.3)
p  = plot(f, (x, 0, pi), color='blue')
p += plot(f, (x, 0, x1), color='blue', fill=True, fillcolor='blue', fillalpha=0.25)

p += plot(g, (x, 0, pi), color='red')
p += plot(g, (x, 0, x1), color='red', fill=True, fillcolor='red', fillalpha=0.25)
p.show()

In [ ]:
print((f-g).find_root(2,2.3)) # Numerische Suche nach Schnittstellen von f und g
print((f-g).find_root(2.3,3))
print((f-g).find_root(3,3.5))

In [ ]:
(f-g).roots() # findet nur x=0 als Nullstelle
# die anderen kommen von der transzendenten Gleichung sin(x)=-sin(x²), kein Polynom, symbolisch nicht lösbar

In [ ]:
# Symbolische Flächenbestimmung
I = integrate(f-g, x, pi/4, pi/2)
print(f"Fläche: {I} ≈ {float(I)}")

---
## Gleichungssysteme

**Schnitt von zwei Geraden:** $2x+y=4$, $x+2y=4$:

In [ ]:
x, y = var('x y')
solve([2*x+y==4,x+2*y==4],x,y) # (4/3, 4/3)

Oder etwas umständlicher:
$$A = \begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}, \quad b = \begin{pmatrix} 4 \\ 4 \end{pmatrix}, \quad Ax = b \;\Rightarrow\; x = A^{-1} b$$

In [ ]:
A = matrix([[2,1],[1,2]])
b = vector([4,4])
sol = A.inverse() * b # (4/3, 4/3)
print(f"sol: {sol}")

x = var('x')
p  = plot(4 - 2*x, (x, -0.5, 3), color='blue',   legend_label='2x+y=4')
p += plot(2 - x/2, (x, -0.5, 4.5), color='orange', legend_label='x+2y=4')
p += point(sol, size=50, color='red', zorder=5)
p.show(aspect_ratio=1, xmin=-0.5, xmax=4.5, ymin=-0.5, ymax=4.5)

**Schnitt von zwei Kreisen:**

$$C_1: (x-1)^2 + y^2 = 4 \qquad C_2: (x+1)^2 + y^2 = 4$$

In [ ]:
x, y = var('x y')
C1 = (x-1)^2 + y^2 == 4
C2 = (x+1)^2 + y^2 == 4

In [ ]:
solutions = solve([C1, C2], x, y) # Symbolische Schnittpunkte
for s in solutions:
    sx = s[0].rhs()
    sy = s[1].rhs()
    print(f"{s} -> ({sx},{sy}) ≈ ({RR(sx)},{RR(sy)})")

In [ ]:
p  = implicit_plot(C1.lhs() - C1.rhs(), (x, -4, 4), (y, -3, 3), color='blue')
p += implicit_plot(C2.lhs() - C2.rhs(), (x, -4, 4), (y, -3, 3), color='red')
for s in solutions:
    sx = s[0].rhs()
    sy = s[1].rhs()
    p += point((RR(sx), RR(sy)), size=50, color='black', zorder=2)
p.show(aspect_ratio=1)

**Idee hinter symbolischen Lösung:**

In [ ]:
# jeder Schnittpunkt (x,y) ist Nullstelle beider Gleichungen 
f1 = (x-1)^2 + y^2 - 4 
f2 = (x+1)^2 + y^2 - 4 

In [ ]:
# ... und Nullestelle von f3 = f1-f2
f3 = (f1-f2)
print(f3) 
print(f3.simplify_full()) # 4x ist nur Null wenn x=0 

In [ ]:
# ... und Nullstelle von f4 = f1+f2, insbesondere wenn x=0 gesetzt
f4 = (f1+f2)
print(f4)
print(f4(x=0))
print(f4(x=0).roots())

**Anmerkung**: reelle Lösungen zu finden ist im Allgemeinen schwer (zwischen NP und PSPACE)
[https://en.wikipedia.org/wiki/Existential_theory_of_the_reals](https://en.wikipedia.org/wiki/Existential_theory_of_the_reals)

---
# Geometrie

2D und 3D Strukturen mit wenig Code visualisieren

**Beispiel:** 

In [ ]:
n = 100
point2d([(cos(x*2*pi/n),sin(x*2*pi/n)) for x in range(n)]).show(aspect_ratio=1)

In [ ]:
p = point3d([(cos(x*2*pi/n),sin(x*2*pi/n),0) for x in range(n)],color='blue')
p += point3d([(cos(x*2*pi/n),0,sin(x*2*pi/n)) for x in range(n)],color='red')
p += point3d([(0,cos(x*2*pi/n),sin(x*2*pi/n)) for x in range(n)],color='green')
p.show(aspect_ratio=1)

In [ ]:
import random
from scipy.spatial import Voronoi, ConvexHull

def new_points():
    return [(random.uniform(0,1), random.uniform(0,1)) for _ in range(20)]

pts = new_points()
hull2d = ConvexHull(pts)
hull_indices = set(hull2d.vertices)

@interact
def geo_demo(mode=selector(['Nearest Neighbor', 'Voronoi', 'Convex Hull'], label='Modus')):
    global pts, hull2d, hull_indices
    hull2d = ConvexHull(pts)
    hull_indices = set(hull2d.vertices)

    inner = [pts[i] for i in range(len(pts)) if i not in hull_indices]
    outer = [pts[i] for i in hull_indices]

    p = points(inner, size=30, color='black', zorder=5)
    p += points(outer, size=30, color='blue', zorder=5)

    if mode == 'Nearest Neighbor':
        for i, (ax, ay) in enumerate(pts):
            best = min((j for j in range(len(pts)) if j != i),
                       key=lambda j: (pts[j][0]-ax)**2 + (pts[j][1]-ay)**2)
            bx, by = pts[best]
            p += arrow((ax, ay), (bx, by), color='blue', width=1, arrowsize=2)

    elif mode == 'Voronoi':
        vor = Voronoi(pts)
        for simplex in vor.ridge_vertices:
            if -1 not in simplex:
                x0, y0 = vor.vertices[simplex[0]]
                x1, y1 = vor.vertices[simplex[1]]
                p += line([(x0,y0),(x1,y1)], color='red')

    elif mode == 'Convex Hull':
        hull_pts = [pts[i] for i in hull2d.vertices]
        p += polygon(hull_pts, color='lightblue', alpha=0.5)
        p += line(hull_pts+[hull_pts[0]], color='blue', thickness=2)

    p.show(axes=False, frame=True, xmin=-0.05, xmax=1.05, ymin=-0.05, ymax=1.05)

In [ ]:
# viele geometrische Formen wie Ikosaeder verfügbar
I = polytopes.icosahedron()
print("Ecken:", len(I.vertices()))
print("Flächen:", len(I.faces(2)))
I.plot()

In [ ]:
import random
from scipy.spatial import ConvexHull
import numpy as np

random.seed()  # system time
pts3 = [(random.uniform(-1,1), random.uniform(-1,1), random.uniform(-1,1)) for _ in range(30)]
arr3 = np.array(pts3)

hull3 = ConvexHull(arr3)
hull_idx3 = set(hull3.vertices)

p3 = point3d([pts3[i] for i in hull_idx3], size=10, color='blue')
p3 += point3d([pts3[i] for i in range(len(pts3)) if i not in hull_idx3], size=10, color='black')

for simplex in hull3.simplices:
    tri = [pts3[i] for i in simplex]
    p3 += polygon3d(tri, color='lightblue', alpha=0.3)

p3.show()

**3D Pseudohyperebenen-Arrangement:**

[lasagne3d/example_pshyperplanes.html](lasagne3d/example_pshyperplanes.html)

[https://helenabergold.github.io/supp/3d_signotopes/nonextendable_sign48_pshyperplane.html](https://helenabergold.github.io/supp/3d_signotopes/nonextendable_sign48_pshyperplane.html)


### Arrangements in der Ebene und auf der Sphäre 

<img src="figs/apc_2019_p11.png" width="80%">



* Geometrische Objekte durch Graphen repräsentieren
* Koordinaten irrelevant, nur Kombinatorik/Topologie erfasst
* "Gleichheitstest" via Graphenisomorphie
* Eigenschaften wie "Anzahl der Dreiecks-Zellen" in Graph sichtbar

<img src=figs/arrangement_graphs.png width=80%>

**Primal-Graph:** Schnittpunkte → Knoten, Bögen → Kanten

**Dual-Graph:** Zellen → Knoten, gemeinsame Kante → Kante

**Primal-Dual:** Kombination beider Sichten

<img src="figs/apc_2019_p10.png" width="80%">

Visualisierung im IPE-Format — erlaubt manuelles Überarbeiten und Export als PDF.

(Direkter SVG/PDF-Export aus Sage möglich, aber Nachbearbeitung aufwändiger.)

[IPE Drawing Editor](https://ipe.otfried.org/) ([im Browser ausprobieren](https://ipe-web.otfried.org/index.html))


**Gegenbeispiel zu Grünbaum's Vermutung aus 1970ern aus Computer**:

<img src=figs/arrangement.png width=50%>


In [ ]:
# Dual-Graph des Arrangements als sparse6-String
s = ":_gBca`GH`GI`HIdGeHfIbKOcJMaLNbMcNaOfPSdQTeRUdSeTfU_WYZ_VY[_XZ["
g = Graph(s)
print(g.edges(labels=False))
g.plot()

In [ ]:
# Planare Darstellung
g.is_planar(set_pos=1)
g.plot()

---
# Graphen

* eigene native Graphbibliothek (Cython/C-Backend)  
* Schnittstellen zu **Boost Graph Library** (C++) und **NetworkX** (Python).

In [ ]:
# Graph definieren und plotten
G = Graph({'A': ['B','C','D'], 'B': ['C','E'], 'C': ['F'], 'D': ['E'], 'E': ['F'], 'F': []})
print("Knoten:", G.vertices())
print("Kanten:", G.edges(labels=False))
print("planar:", G.is_planar(set_pos=True)) # planare Zeichnung falls vorhanden, optional
G.plot()

In [ ]:
# Graphfärbung
chi = G.chromatic_number()
print("Chromatische Zahl:", chi)

coloring = G.coloring()
print("Färbung:", coloring)

G.plot(vertex_colors={
    'red':   coloring[0],
    'blue':  coloring[1],
    'green': coloring[2] if len(coloring) > 2 else []
})

In [ ]:
# paarweise Kürzeste Wege
algo = choice(["BFS","Floyd-Warshall-Cython","Floyd-Warshall-Python","Dijkstra_NetworkX","Dijkstra_Boost","Johnson_Boost"])
# Liste via "G.distance_all_pairs?"

print("Algorithmus:",algo)
D = G.distance_all_pairs(algorithm=algo)
vertices = sorted(G.vertices())

print("Distanzmatrix:")
header = "    " + "  ".join(f"{v:>3}" for v in vertices)
print(header)
for u in vertices:
    row = f"{u:>3} " + "  ".join(f"{D[u][v]:>3}" for v in vertices)
    print(row)

---
# Lineare Programmierung (LP)

mathematisches Verfahren zur Optimierung einer linearen Zielfunktion unter Einhaltung linearer Nebenbedingungen

**Beispiel**
$$\max\; x + y$$

$$\begin{align}
\text{s.t.} \quad 2x + y &\leq 4 \quad (1)\\
x + 2y &\leq 4 \quad (2)\\
x, y &\geq 0
\end{align}$$


In [ ]:
p = MixedIntegerLinearProgram(maximization=True)
x = p.new_variable(nonnegative=True)

p.set_objective(x[1] + x[2])
p.add_constraint(2*x[1] +   x[2] <= 4)
p.add_constraint(  x[1] + 2*x[2] <= 4)

p.solve()
print(f"x = {RR(p.get_values(x[1]))}")
print(f"y = {RR(p.get_values(x[2]))}")
print(f"z (zielfunktion) = {RR(p.get_objective_value())}")

In [ ]:
# graphische Darstellung der Bedingungen und Lösungsraum
x, y = var('x y')

r1 = region_plot(2*x + y <= 4, (x, -0.5, 3), (y, -0.5, 3), incol='lightblue', alpha=0.4)
r2 = region_plot(x + 2*y <= 4, (x, -0.5, 3), (y, -0.5, 3), incol='lightyellow', alpha=0.4)
r3 = region_plot([x >= 0, y >= 0, 2*x + y <= 4, x + 2*y <= 4], (x,-0.5,3), (y,-0.5,3), incol='lightgreen', alpha=0.5)

l1 = implicit_plot(2*x + y - 4, (x, -0.5, 3), (y, -0.5, 3), color='blue')
l2 = implicit_plot(x + 2*y - 4, (x, -0.5, 3), (y, -0.5, 3), color='orange')

zline   = implicit_plot(x + y - 8/3, (x, -0.5, 3), (y, -0.5, 3), color='red', linestyle='--', linewidth=2)
opt     = point((4/3, 4/3), size=50, color='red', zorder=5)
corners = points([(0,0),(2,0),(4/3,4/3)], size=40, color='black', zorder=5)
path    = line([(0,0),(2,0),(4/3,4/3)], color='black', linestyle='--', thickness=1.5)

p = r1+r2+r3+l1+l2+zline+opt+corners+path
p += text("(1): 2x+y≤4",    (1.1, 2.5),          color='blue',       fontsize=10)
p += text("(2): x+2y≤4",    (2.5, 1),          color='darkorange', fontsize=10)
p += text("z*=(4/3, 4/3)",  (4/3+0.3, 4/3+0.1), color='red',        fontsize=9)
p.show(aspect_ratio=1,xmin=0,ymin=0)

**Simplex Verfahren:** wandert entlang Ecken des zulässigen Bereichs bis Optimum erreicht:
$$(0,0) \;\to\; (2,0) \;\to\; z^* = (4/3,\, 4/3)$$

Rote Linie durch $z^*$ zeigt optimalität bzgl. Zielfunktion $z = x+y$.

**Anmerkungen:**
- Anzahl Ecken wächst exponentiell mit Anzahl der Variablen (Dimensionen)
- **Simplex-Verfahren**:
    -  erwartungsgemäß polynomiell
    - in Praxis Millionen von Variablen und Constraints möglich
    - im worst-case exponentiell
- **Innere-Punkte-Methode**: (schwach) polynomiell
- GLPK (open) vs Gurobi (kommerziell)

---
# Mixed Integer Linear Programming (MILP)

**Beispiel: Gewinnmaximierung im Stall**

Variablen: $x_1$ = Schweine, $x_2$ = Hühner, $x_3$ = Kühe, 
jeweils **ganzzahlig, nicht-negativ**.

$$\max \; 30 x_1 + 5 x_2 + 50 x_3$$

$$\begin{align}
\text{s.t.} \quad
\underbrace{50 x_1 + 10 x_2 + 200 x_3}_{\text{Wasser (L/Tag)}} &\leq 400 \\
\underbrace{3 x_1 + 0.5 x_2 + 8 x_3}_{\text{Futter (kg/Tag)}}  &\leq 40 \\
\underbrace{x_1 + 0.1 x_2 + x_3}_{\text{Stallplätze}}           &\leq 7
\end{align}$$

$$x_1, x_2, x_3 \in \mathbb{Z}_{\geq 0}$$

In [ ]:
p = MixedIntegerLinearProgram(maximization=True)
x = p.new_variable(integer=1, nonnegative=True)

p.set_objective(30*x[1] + 5*x[2] + 50*x[3])          # max Profit (€)

p.add_constraint(50*x[1] + 10*x[2] + 200*x[3] <= 400)  # Wasser (L/Tag)
p.add_constraint( 3*x[1] +  0.5*x[2] +  8*x[3] <= 40)   # Futter (kg/Tag)
p.add_constraint( 1*x[1] +  0.1*x[2] +    1*x[3] <= 7)      # Stallplätze

p.solve()

print(f"Schweine: {RR(p.get_values(x[1]))}")
print(f"Hühner:   {RR(p.get_values(x[2]))}")
print(f"Kühe:     {RR(p.get_values(x[3]))}")
print(f"Profit:   {RR(p.get_objective_value())}")

---
## Encoding & Solver-Effizienz

- Modellierung oft **straight-forward**
- Integer-Constraints → **NP-schwer**
- Encoding-Entscheidungen können Laufzeit von Solver **drastisch** beeinflussen
    - gute Heuristik: wenig Variablen & Constraints 
    - **Hilfsvariablen** zur Verkleinerung (Variablen-Constraints-Tradeoff)
    - **Symmetrien** im Suchraum werden von Solvern meist *nicht* erkannt → manuell brechen
- **SAT** ist Spezialfall von MILP:
    - nur 0/1-Variablen,
    - logisches AND/OR als Summen 

---
## SAT (Boolean Satisfiability) vs IP (Integer Programming)


**SAT-Instanz in IP-Instanz umformen (straight-forward)** 

- Boolsche Variablen $x_i \in \{0,1\}$

- Boolean Constraints als Summen:
$$x_1 \lor (1-x_2) \lor x_3 \quad\Leftrightarrow\quad x_1 + (1-x_2) + x_3 \geq 1 \quad\Leftrightarrow\quad x_1 -x_2 + x_3 \geq 0$$

**IP-Instanz in SAT-Instanz umformen**:
- Summen bzw. Teilsummen mittels Indikatorvariablen beschreiben

Beispiel: $x_1+x_2+x_3+x_4$
* Teilsumme $u := x_1+x_2$
    * 3 mögliche Werte: 0, 1, oder 2 
    * 3 Indikatorvariablen "$u$ ist 0", "$u$ ist 1", "$u$ ist 2"
* Teilsumme $v := u+x_3$:
    * 4 mögliche Werte: 0 bis 3
    * 4 Indikatorvariablen
* Teilsumme $w := v+x_4$:
    *  5 mögliche Werte: 0 bis 4
    *  5 Indikatorvariablen
* Gesamt: $3+4+5$ Indikatorvariablen

 **pysat** : simple and unified interface to a number of state-of-the-art Boolean satisfiability (SAT) solvers like CaDiCaL and Kissat.

In [ ]:
!sage -pip install python-sat # installiere python-sat 

In [ ]:
from pysat.solvers import Cadical195

# Formel: (x1 ∨ x2) ∧ (¬x1 ∨ ¬x2)
cnf = [[1, 2],[-1, -2]]

def convert2ints(cnf): 
    return [[int(x) for x in c] for c in cnf] 
    # Sage Integers -> python int conversion
    # In Sage ist 1 kein int (so wie in python) sondern ein Integer

solver = Cadical195(bootstrap_with=convert2ints(cnf))
for model in solver.enum_models():
    print(model)

## Beispiel: Ramsey-Numer $R(3,3) = 6$

Aussage: In jeder 2-Färbung der Kanten vom vollständigen Graphen $K_6$ gibt es ein einfarbiges Dreieck.

Beweis per SAT: teste ob $R(3,3) > n$ lösbar ist.

In [ ]:
graphs.CompleteGraph(6).plot()

In [ ]:
from itertools import combinations
from pysat.solvers import Cadical195
from pysat.formula import IDPool

def edge_var(u,v):
    return pool.id((min(u,v),max(u,v))) # Variable codiert Kante von u nach v 

def ramsey_gt(n, k, l):
    """Ist R(k,l) > n? Liefert (Modell, pool) oder None."""
    V = range(n)
    pool = IDPool()

    cnf = []
    for a,b,c in combinations(V, 3):
        cnf.append([-edge_var(a,b), -edge_var(a,c), -edge_var(b,c)])
        cnf.append([+edge_var(a,b), +edge_var(a,c), +edge_var(b,c)])

    solver = Cadical195(bootstrap_with=cnf)
    if solver.solve():
        model = solver.get_model()
        return model,pool
    else:
        return None,pool

for n in [5, 6]:
    model,pool = ramsey_gt(n, 3, 3)
    print(f'R(3,3) > {n}: {"SAT (Färbung existiert)" if model else "UNSAT"}')
    if model:
        red_edges  = [(u,v) for u,v in combinations(V,2) if  edge_var(u,v) in model]
        blue_edges = [(u,v) for u,v in combinations(V,2) if -edge_var(u,v) in model]

        G = Graph([V, red_edges + blue_edges])
        pos = G.layout('circular')
        
        G.plot(pos=pos, vertex_size=300, edge_colors={'red': red_edges, 'blue': blue_edges}).show()

---
## SMT — Satisfiability Modulo Theories

SAT + Theorien: Arithmetik, Arrays, Bitvektoren, ...

**Anwendung:** Formale Verifikation von Algorithmen (z.B. Bubblesort korrektheit)

**Beispiel**: Ist $x^2 + y^2 = 1 \land x = y \land x > 0$ erfüllbar? (Einheitskreis schneidet Gerade $y=x$)


In [ ]:
!sage -pip install z3-solver # installiere z3 solver

In [ ]:
from z3 import *
x, y = Reals('x y')
s = Solver()
s.add(x**2 + y**2 == 1, x == y, x > 0)
print("satisfiable:",s.check())
print("model:",s.model())

---

<img src="figs/apc_circ_2018_last.png" width="70%">

Fragen? Code & Notebook: <a href="https://github.com/manfredscheucher/talk-glt26">https://github.com/manfredscheucher/talk-glt26</a>